# EDA longitudinal ENIGH 2018-2024

Este notebook conserva los displays de las bases originales y de las tablas homologadas/apiladas. Después de esos displays continúo con el EDA desde variables numéricas en adelante; la revisión de estabilidad quedó separada en `estabilidad_de_bases.ipynb`.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 90)
pd.set_option("display.max_rows", 90)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid", context="notebook")


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("No pude localizar la raiz del proyecto.")


ROOT = find_project_root()
RAW_ROOT = ROOT / "data/raw/EINGH"
INTERIM_ROOT = ROOT / "data/interim/revision_3"
YEARS = [2018, 2020, 2022, 2024]
CORE_TABLES = ["concentradohogar", "hogares", "ingresos", "poblacion", "trabajos", "viviendas"]
STACK_PATHS = {name: INTERIM_ROOT / f"{name}_decodificada_2018_2024.csv.gz" for name in CORE_TABLES}

CATALOG_PATH = INTERIM_ROOT / "catalogo_categoricas_enigh.csv"
INVENTORY_PATH = INTERIM_ROOT / "inventario_variables_revision_3.csv"
MANUAL_REVIEW_PATH = INTERIM_ROOT / "variables_revision_manual.csv"
VALIDATION_PATH = INTERIM_ROOT / "validacion_mappings_categoricas.csv"

METADATA = pd.read_csv(ROOT / "docs/enigh_variable_metadata.csv", dtype=str)
CATALOG = pd.read_csv(CATALOG_PATH, dtype=str)
VARIABLE_INVENTORY = pd.read_csv(INVENTORY_PATH, dtype=str)
MANUAL_REVIEW = pd.read_csv(MANUAL_REVIEW_PATH, dtype=str)
MAPPING_VALIDATION = pd.read_csv(VALIDATION_PATH, dtype=str)
VARIABLE_INVENTORY = (
    pd.concat([VARIABLE_INVENTORY, MANUAL_REVIEW], ignore_index=True)
    .drop_duplicates(["tabla", "variable"], keep="last")
)

MISSING_LABEL = "Sin informacion"


def render_figure(fig):
    display(fig)

print(f"Proyecto: {ROOT}")
print(f"Fuente de datos para EDA: {INTERIM_ROOT}")


## Bases originales

Primero cargo y muestro las primeras 20 filas de las bases originales usadas para construir las tablas homologadas.

In [ ]:
original_samples = {}
original_sample_rows = []

for table in CORE_TABLES:
    for year in YEARS:
        path = RAW_ROOT / str(year) / f"{table}.csv"
        df = pd.read_csv(path, nrows=20, low_memory=False, encoding="utf-8-sig")
        original_samples[(table, year)] = df
        original_sample_rows.append({
            "tabla": table,
            "anio": year,
            "filas_muestra": len(df),
            "columnas_muestra": df.shape[1],
            "primeras_columnas": ", ".join(df.columns[:8]),
        })

display(pd.DataFrame(original_sample_rows))


## Tablas decodificadas y apiladas


In [ ]:
tables = {}
table_load_rows = []

for table in CORE_TABLES:
    df = pd.read_csv(STACK_PATHS[table], low_memory=False)
    tables[table] = df
    globals()[table] = df
    table_load_rows.append({
        "tabla": table,
        "archivo": STACK_PATHS[table].name,
        "filas": len(df),
        "columnas": df.shape[1],
        "columnas_desc": sum(col.endswith("_desc") for col in df.columns),
    })

display(pd.DataFrame(table_load_rows))


## Preparación mínima para el EDA

Reconstruyo únicamente los objetos necesarios para los análisis posteriores. La estabilidad y las exclusiones se revisan en el notebook separado.

In [ ]:
def clean_for_eda(df):
    clean = df.copy()
    for col in clean.columns:
        if pd.api.types.is_object_dtype(clean[col]) or str(clean[col].dtype).startswith("string"):
            clean[col] = clean[col].astype("string").str.strip()
            clean[col] = clean[col].replace({"": pd.NA, "NA": pd.NA, "N/A": pd.NA})
    return clean


def display_col(df, var):
    desc = f"{var}_desc"
    return desc if desc in df.columns else var


def presentation_series(df, var, missing_label=MISSING_LABEL):
    col = display_col(df, var)
    return df[col].astype("string").fillna(missing_label)


def conceptual_type(table, var):
    base = var[:-5] if var.endswith("_desc") else var
    rows = VARIABLE_INVENTORY[
        (VARIABLE_INVENTORY["tabla"] == table)
        & (VARIABLE_INVENTORY["variable"] == base)
    ]
    if rows.empty:
        return None
    return rows.iloc[0]["tipo_conceptual"]


def is_numeric_analysis_var(table, var):
    if var.endswith("_desc"):
        return False
    tipo = conceptual_type(table, var)
    return isinstance(tipo, str) and tipo.startswith("num")


def compact_values(values, limit=12):
    vals = [str(v) for v in pd.Series(values).dropna().drop_duplicates().tolist()]
    shown = vals[:limit]
    suffix = "" if len(vals) <= limit else f" ... (+{len(vals) - limit})"
    return "; ".join(shown) + suffix


def category_order(table, var, observed_values):
    observed = [str(v) for v in pd.Series(observed_values).dropna().drop_duplicates().tolist()]
    if not observed:
        return observed

    sub = CATALOG[(CATALOG["tabla"] == table) & (CATALOG["variable"] == var)].copy()
    if sub.empty or "orden_categoria" not in sub.columns:
        return observed

    sub["_orden"] = pd.to_numeric(sub["orden_categoria"], errors="coerce")
    sub = sub.sort_values(["_orden", "codigo"], na_position="last")

    ordered = []
    label_order = sub["etiqueta"].dropna().astype(str).drop_duplicates().tolist()
    code_order = sub["codigo"].dropna().astype(str).drop_duplicates().tolist()

    for candidate_order in [label_order, code_order]:
        matches = [value for value in candidate_order if value in observed]
        if matches:
            ordered.extend(matches)
            break

    ordered.extend([value for value in observed if value not in ordered])
    return ordered


clean_tables = {table: clean_for_eda(df) for table, df in tables.items()}


## 5. Variables numéricas

In [ ]:
NUMERIC_RELEVANT = {
    "concentradohogar": ["ing_cor", "ingtrab", "trabajo", "sueldos", "negocio", "hospital", "gasto_mon", "tot_integ", "ocupados", "percep_ing", "edad_jefe"],
    "ingresos": ["ing_tri"],
    "poblacion": ["edad", "grado", "gradoaprob", "num_trabaj", "hijos_viv", "hijos_mue", "hijos_sob"],
    "trabajos": ["htrab"],
    "hogares": ["num_carret", "num_pickup", "num_auto", "num_compu"],
    "viviendas": ["num_cuarto", "cuart_dorm", "tot_resid", "renta"],
}

def numeric_stats(df, cols, by_year=False):
    rows = []
    for col in [c for c in cols if c in df.columns]:
        groups = df.groupby("anio") if by_year else [(None, df)]
        for year, g in groups:
            s = pd.to_numeric(g[col], errors="coerce")
            rows.append({
                "anio": year,
                "variable": col,
                "count": s.notna().sum(),
                "mean": s.mean(),
                "median": s.median(),
                "std": s.std(),
                "p25": s.quantile(.25),
                "p75": s.quantile(.75),
                "p95": s.quantile(.95),
                "p99": s.quantile(.99),
                "min": s.min(),
                "max": s.max(),
                "zero_pct": (s == 0).mean() * 100,
                "negative_pct": (s < 0).mean() * 100,
            })
    return pd.DataFrame(rows)

numeric_stats_all = {table: numeric_stats(clean_tables[table], cols) for table, cols in NUMERIC_RELEVANT.items()}
numeric_stats_year = {table: numeric_stats(clean_tables[table], cols, by_year=True) for table, cols in NUMERIC_RELEVANT.items()}

for table, stats in numeric_stats_all.items():
    print("\n" + "=" * 90)
    print(table)
    display(stats.round(2))

In [ ]:
for table, cols in NUMERIC_RELEVANT.items():
    df = clean_tables[table]
    for col in [c for c in cols if c in df.columns][:6]:
        s = pd.to_numeric(df[col], errors="coerce")
        if s.notna().sum() < 100:
            continue
        upper = s.quantile(.99)
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        sns.histplot(s[s <= upper], bins=45, kde=True, ax=axes[0], color="#4C78A8")
        axes[0].set_title(f"{table}.{col}: histograma hasta P99")
        sns.boxplot(data=df.assign(_v=s.clip(upper=upper)), x="anio", y="_v", showfliers=False, ax=axes[1], color="#A0CBE8")
        axes[1].set_title(f"{table}.{col}: boxplot por año")
        axes[1].set_xlabel("Año")
        axes[1].set_ylabel(col)
        plt.tight_layout()
        render_figure(fig)
        plt.close(fig)

In [ ]:
for table, stats in numeric_stats_year.items():
    keep = stats[stats["variable"].isin(NUMERIC_RELEVANT[table][:6])]
    if keep.empty:
        continue
    display(keep.round(2).head(60))
    for col in keep["variable"].unique()[:4]:
        temp = keep[keep["variable"] == col]
        fig, ax = plt.subplots(figsize=(7, 3.5))
        sns.lineplot(data=temp, x="anio", y="median", marker="o", label="mediana", ax=ax)
        sns.lineplot(data=temp, x="anio", y="mean", marker="o", label="media", ax=ax)
        ax.set_title(f"{table}.{col}: centro por año")
        ax.set_xticks(YEARS)
        plt.tight_layout()
        render_figure(fig)
        plt.close(fig)

## 6. Variables categoricas

Las salidas visibles usan la columna descriptiva `*_desc` cuando ya existe en `revision_3`; los codigos originales permanecen en las bases para trazabilidad.


In [ ]:
CATEGORICAL_RELEVANT = {
    "concentradohogar": ["tam_loc", "est_socio", "clase_hog", "sexo_jefe", "educa_jefe"],
    "hogares": ["telefono", "celular", "conex_inte", "tarjeta"],
    "poblacion": ["sexo", "nivelaprob", "edo_conyug", "trabajo_mp"],
    "trabajos": ["subor", "indep", "contrato", "tiene_suel", "tipoact"],
    "viviendas": ["tipo_viv", "combustible", "medidor_luz", "tenencia", "drenaje", "disp_elect"],
}

categorical_validation_rows = []
cat_summary_rows = []
cat_year_tables = {}

for table, cols in CATEGORICAL_RELEVANT.items():
    df = clean_tables[table]
    for col in [c for c in cols if c in df.columns]:
        visual_col = display_col(df, col)
        has_desc = visual_col != col
        code_series = df[col].astype("string")
        label_series = df[visual_col].astype("string") if has_desc else pd.Series(pd.NA, index=df.index, dtype="string")

        if has_desc:
            code_label_pairs = pd.DataFrame({"codigo": code_series, "label": label_series}).drop_duplicates()
            missing_codes = (
                code_label_pairs.loc[code_label_pairs["codigo"].notna() & code_label_pairs["label"].isna(), "codigo"]
                .drop_duplicates()
                .astype(str)
                .tolist()
            )
            status = "Con _desc" if not missing_codes else "Con _desc; revisar codigos sin label"
        else:
            missing_codes = code_series.dropna().drop_duplicates().astype(str).tolist()
            status = "Pendiente sin _desc"

        categorical_validation_rows.append({
            "tabla": table,
            "variable_original": col,
            "variable_descriptiva": visual_col if has_desc else pd.NA,
            "disponible": "Si" if has_desc else "No",
            "codigos_distintos": code_series.nunique(dropna=True),
            "labels_distintos": label_series.nunique(dropna=True) if has_desc else pd.NA,
            "codigos_sin_label": compact_values(missing_codes),
            "estado": status,
        })

        s = presentation_series(df, col)
        vc = s.value_counts(dropna=False, normalize=True)
        dominant = vc.index[0] if not vc.empty else pd.NA
        cat_summary_rows.append({
            "tabla": table,
            "variable": col,
            "variable_visual": visual_col,
            "categorias": s.nunique(dropna=True),
            "dominante": dominant,
            "dominante_pct": vc.iloc[0] * 100 if not vc.empty else np.nan,
            "missing_pct": df[visual_col].isna().mean() * 100 if has_desc else df[col].isna().mean() * 100,
            "categorias_menor_1pct": int((s.value_counts(normalize=True) < .01).sum()),
        })

        tab = pd.crosstab(df["anio"], s, normalize="index") * 100
        tab = tab.reindex(columns=category_order(table, col, tab.columns))
        cat_year_tables[(table, col)] = tab

categorical_validation = pd.DataFrame(categorical_validation_rows)
display(categorical_validation)

categorical_pending = categorical_validation[categorical_validation["estado"] != "Con _desc"][
    ["tabla", "variable_original", "codigos_sin_label", "estado"]
].copy()
display(categorical_pending)

cat_summary = pd.DataFrame(cat_summary_rows)
display(cat_summary.round(2))


In [ ]:
for (table, col), tab in list(cat_year_tables.items()):
    top = tab.mean().sort_values(ascending=False).head(8).index
    top = category_order(table, col, top)
    display(tab[top].round(2))
    plot_df = tab[top].reset_index().melt(id_vars="anio", var_name="categoria", value_name="pct")
    visual_col = display_col(clean_tables[table], col)
    fig, ax = plt.subplots(figsize=(9, 4))
    sns.barplot(data=plot_df, x="anio", y="pct", hue="categoria", hue_order=top, ax=ax)
    ax.set_title(f"{table}.{col}: composicion por ano")
    ax.set_xlabel("Ano")
    ax.set_ylabel("%")
    ax.legend(title=visual_col, bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    render_figure(fig)
    plt.close(fig)


## 7. Ingresos

In [ ]:
income_meta = METADATA[
    (METADATA["table"].isin(["concentradohogar.csv", "ingresos.csv"]))
    & (METADATA["variable"].isin(["ingtrab", "ing_cor", "trabajo", "sueldos", "negocio", "ing_tri"]))
][["year", "table", "variable", "label", "dtype"]].drop_duplicates()
income_meta.sort_values(["table", "variable", "year"])

In [ ]:
income_df = clean_tables["concentradohogar"].copy()
income_df["ingtrab"] = pd.to_numeric(income_df["ingtrab"], errors="coerce")
income_df["log1p_ingtrab"] = np.log1p(income_df["ingtrab"].clip(lower=0))
income_p99 = income_df["ingtrab"].quantile(.99)

income_summary = numeric_stats(income_df, ["ingtrab"])
income_year = numeric_stats(income_df, ["ingtrab"], by_year=True)
display(income_summary.round(2))
display(income_year.round(2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
sns.histplot(income_df.loc[income_df["ingtrab"] <= income_p99, "ingtrab"], bins=60, kde=True, ax=axes[0], color="#4C78A8")
axes[0].set_title("ingtrab hasta P99")
sns.histplot(income_df["log1p_ingtrab"], bins=60, kde=True, ax=axes[1], color="#59A14F")
axes[1].set_title("log1p(ingtrab)")
sns.boxplot(data=income_df.assign(_v=income_df["ingtrab"].clip(upper=income_p99)), x="anio", y="_v", showfliers=False, ax=axes[2], color="#A0CBE8")
axes[2].set_title("ingtrab por año")
axes[2].set_ylabel("ingtrab")
plt.tight_layout()
render_figure(fig)
plt.close(fig)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for col in ["median", "p25", "p75", "p95"]:
    sns.lineplot(data=income_year, x="anio", y=col, marker="o", label=col, ax=ax)
ax.set_title("Evolución nominal de ingtrab")
ax.set_xticks(YEARS)
ax.set_xlabel("Año")
ax.set_ylabel("ingtrab")
plt.tight_layout()
render_figure(fig)
plt.close(fig)

## 8. Variables numéricas vs ingresos

In [ ]:
income_target = income_df["ingtrab"]
numeric_income_rows = []
for col in [c for c in clean_tables["concentradohogar"].columns if c not in ["anio", "folioviv", "foliohog", "ingtrab"]]:
    if not is_numeric_analysis_var("concentradohogar", col):
        continue
    x = pd.to_numeric(clean_tables["concentradohogar"][col], errors="coerce")
    if x.notna().mean() < .80 or x.nunique(dropna=True) <= 2:
        continue
    valid = x.notna() & income_target.notna()
    xv, yv = x[valid], income_target[valid]
    numeric_income_rows.append({
        "variable": col,
        "Pearson": xv.corr(yv),
        "Spearman": xv.rank().corr(yv.rank()),
        "n": int(valid.sum()),
    })
income_correlations = pd.DataFrame(numeric_income_rows)
income_correlations["abs_spearman"] = income_correlations["Spearman"].abs()
income_correlations = income_correlations.sort_values("abs_spearman", ascending=False)
income_correlations.head(30).round(3)


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(data=income_correlations.head(20), y="variable", x="Spearman", color="#F28E2B", ax=ax)
ax.axvline(0, color="black", linewidth=.8)
ax.set_title("Correlación Spearman con ingtrab")
ax.set_xlabel("Spearman")
ax.set_ylabel("")
plt.tight_layout()
render_figure(fig)
plt.close(fig)

In [ ]:
top_scatter = income_correlations.head(4)["variable"].tolist()
scatter = income_df[["ingtrab"] + top_scatter].apply(pd.to_numeric, errors="coerce").dropna().sample(n=5000, random_state=42)
scatter["log1p_ingtrab"] = np.log1p(scatter["ingtrab"].clip(lower=0))
for col in top_scatter:
    tmp = scatter[[col, "log1p_ingtrab"]].dropna().copy()
    tmp["bin"] = pd.qcut(tmp[col].rank(method="first"), q=20, duplicates="drop")
    binned = tmp.groupby("bin", observed=True).agg(x_med=(col, "median"), y_med=("log1p_ingtrab", "median")).reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(7, 4.2))
    sns.scatterplot(data=tmp, x=col, y="log1p_ingtrab", alpha=.15, s=8, edgecolor=None, ax=ax)
    sns.lineplot(data=binned, x="x_med", y="y_med", color="red", marker="o", ax=ax)
    ax.set_title(f"{col} vs log1p(ingtrab)")
    plt.tight_layout()
    render_figure(fig)
    plt.close(fig)

## 9. Variables categóricas vs ingresos

In [ ]:
income_cats = [c for c in CATEGORICAL_RELEVANT["concentradohogar"] if c in income_df.columns]
cat_income_rows = []

for col in income_cats:
    visual_col = display_col(income_df, col)
    keep_cols = [col, "ingtrab"] + ([] if visual_col == col else [visual_col])
    tmp = income_df[keep_cols].copy()
    tmp["_categoria"] = tmp[visual_col].astype("string").fillna(MISSING_LABEL)
    tmp = tmp.dropna(subset=["ingtrab"])
    grouped = (
        tmp.groupby("_categoria")["ingtrab"]
        .agg(n="count", mean="mean", median="median", p25=lambda x: x.quantile(.25), p75=lambda x: x.quantile(.75))
        .reset_index()
        .rename(columns={"_categoria": "categoria"})
    )
    grouped["tabla"] = "concentradohogar"
    grouped["variable"] = col
    grouped["variable_visual"] = visual_col
    cat_income_rows.append(grouped)

cat_income = pd.concat(cat_income_rows, ignore_index=True) if cat_income_rows else pd.DataFrame()
display(cat_income.sort_values(["variable", "median"], ascending=[True, False]).round(2))


In [ ]:
for col in income_cats:
    visual_col = display_col(income_df, col)
    keep_cols = [col, "ingtrab"] + ([] if visual_col == col else [visual_col])
    tmp = income_df[keep_cols].copy()
    tmp["_categoria"] = tmp[visual_col].astype("string").fillna(MISSING_LABEL)
    levels = tmp["_categoria"].value_counts().head(12).index.tolist()
    levels = category_order("concentradohogar", col, levels)
    tmp = tmp[tmp["_categoria"].isin(levels)].copy()
    tmp["_ing"] = tmp["ingtrab"].clip(upper=income_p99)
    fig, ax = plt.subplots(figsize=(10, 4.2))
    sns.boxplot(data=tmp, x="_categoria", y="_ing", order=levels, showfliers=False, color="#A0CBE8", ax=ax)
    ax.set_title(f"ingtrab por {visual_col}")
    ax.set_xlabel(visual_col)
    ax.set_ylabel("ingtrab")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    render_figure(fig)
    plt.close(fig)


## 10. Estabilidad de relaciones por año

In [ ]:
per_year_corr_rows = []
for col in income_correlations.head(12)["variable"]:
    for year, g in income_df.groupby("anio"):
        x = pd.to_numeric(g[col], errors="coerce")
        y = g["ingtrab"]
        valid = x.notna() & y.notna()
        per_year_corr_rows.append({
            "variable": col,
            "anio": year,
            "Pearson": x[valid].corr(y[valid]),
            "Spearman": x[valid].rank().corr(y[valid].rank()),
            "n": int(valid.sum()),
        })
per_year_corr = pd.DataFrame(per_year_corr_rows)
display(per_year_corr.pivot(index="variable", columns="anio", values="Spearman").round(3))

In [ ]:
cat_year_income = []

for col in income_cats:
    visual_col = display_col(income_df, col)
    keep_cols = [col, "anio", "ingtrab"] + ([] if visual_col == col else [visual_col])
    tmp = income_df[keep_cols].dropna(subset=["anio", "ingtrab"]).copy()
    tmp["_categoria"] = tmp[visual_col].astype("string").fillna(MISSING_LABEL)
    top = tmp["_categoria"].value_counts().head(8).index.tolist()
    top = category_order("concentradohogar", col, top)
    tmp = tmp[tmp["_categoria"].isin(top)]
    grouped = tmp.groupby(["anio", "_categoria"])["ingtrab"].median().reset_index()
    grouped["variable"] = col
    grouped["variable_visual"] = visual_col
    cat_year_income.append(grouped.rename(columns={"_categoria": "categoria"}))

cat_year_income = pd.concat(cat_year_income, ignore_index=True) if cat_year_income else pd.DataFrame()
display(cat_year_income.head(80).round(2))


## 11. Correlaciones generales y redundancias

In [ ]:
correlation_blocks = {
    "concentradohogar": ["ing_cor", "ingtrab", "trabajo", "sueldos", "negocio", "gasto_mon", "tot_integ", "ocupados", "percep_ing", "edad_jefe"],
    "poblacion": ["edad", "grado", "gradoaprob", "num_trabaj", "hijos_viv", "hijos_mue", "hijos_sob"],
    "viviendas": ["renta", "num_cuarto", "cuart_dorm", "tot_resid", "tot_hom", "tot_muj", "tot_hog"],
}
for table, cols in correlation_blocks.items():
    cols = [c for c in cols if c in clean_tables[table].columns]
    corr = clean_tables[table][cols].apply(pd.to_numeric, errors="coerce").corr()
    display(corr.round(2))
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, ax=ax)
    ax.set_title(f"Correlaciones generales: {table}")
    plt.tight_layout()
    render_figure(fig)
    plt.close(fig)

## 12. Pairplot

In [ ]:
pair_vars = ["ingtrab", "log1p_ingtrab", "ing_cor", "gasto_mon", "sueldos", "ocupados", "tot_integ", "edad_jefe"]
pair_vars = [c for c in pair_vars if c in income_df.columns]
pair_data = income_df[pair_vars].apply(pd.to_numeric, errors="coerce").dropna()
sample = pair_data.sample(n=min(3000, len(pair_data)), random_state=42)
g = sns.pairplot(sample, corner=True, diag_kind="hist", plot_kws={"alpha": .25, "s": 10, "edgecolor": "none"})
g.fig.suptitle("Pairplot de variables seleccionadas", y=1.02)
render_figure(g.fig)
plt.close(g.fig)